# 🧠 Deep Learning Time-Series Training (CNN & MLP)

Notebook này được sử dụng để huấn luyện 2 mô hình Deep Learning (1D-CNN và Multi-Layer Perceptron) bằng PyTorch.
**Điểm đặc biệt đối với Mạng Nơ-ron:**
- Mạng Nơ-ron không có khả năng hiểu giá trị `NaN` như các thuật toán Cây (LightGBM). Vì vậy bắt buộc phải có bước Pre-processing (FillNA, Clip Infinity).
- Mạng Nơ-ron rất nhạy cảm với thang đo của features. Bắt buộc phải được chuẩn hóa qua hệ thống `StandardScaler`.
- Các mô hình này được train qua chiến lược Time-Series K-Fold.

In [ ]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
import torch
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from src.training.cv_splitter import TimeSeriesCVSplitter
from src.training.metrics import rmspe
from src.models.cnn_model import CNN1DModel
from src.models.mlp_model import MLPModel
from src.training.trainer import NNTrainer

### 1. Nạp Dữ Liệu và Tiền Xử Lý (Imputation & Scaling)

In [ ]:
print("Đang nạp bộ dữ liệu Feature Khổng Lồ (Features_FINAL.feather)...")
df = pd.read_feather('../data/processed/features_FINAL.feather')

# 1. Đặt lại trật tự thời gian t-SNE
cv = TimeSeriesCVSplitter(n_splits=4, random_state=42)
cv.reverse_engineer_time_order(df_train=df, df_features=df)
df['true_time_id'] = cv.recovered_time_order['true_time_id']

# Tách biệt features và các cột metadata
features = [col for col in df.columns if col not in ['time_id', 'stock_id', 'target', 'true_time_id']]
num_features = len(features)
print(f"Có tổng cộng {num_features} Features cho Deep Learning.")

print("Đang thực hiện Data Imputation (Lấp NaN & Vô cực)...")
# Thay vô cực bằng NaN để xử lý chung 1 lần
df[features] = df[features].replace([np.inf, -np.inf], np.nan)

# Lấp các ô giá trị rỗng (NaN) bằng Mean của mỗi cột (Rất quan trọng cho PyTorch)
df[features] = df[features].fillna(df[features].mean())
df[features] = df[features].fillna(0) # Đề phòng cột toàn NaN

print("Đang chuẩn hóa phương sai (StandardScaler)...")
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
print("Hoàn tất Preprocessing!")

### 2. Vòng lặp Huấn Luyện (Training Loop)

In [ ]:
MODELS_DIR = '../models'
os.makedirs(os.path.join(MODELS_DIR, 'cnn'), exist_ok=True)
os.makedirs(os.path.join(MODELS_DIR, 'mlp'), exist_ok=True)

cnn_oof_preds = np.zeros(len(df))
mlp_oof_preds = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(cv.split(df)):
    print(f"\n{'='*25} FOLD {fold+1} {'='*25}")
    
    X_train = df.loc[train_idx, features].values
    y_train = df.loc[train_idx, 'target'].values
    
    X_val = df.loc[val_idx, features].values
    y_val = df.loc[val_idx, 'target'].values
    
    print(f"Kích thước Train: {X_train.shape} | Val: {X_val.shape}")
    
    # --- 1. Huấn luyện 1D-CNN ---
    print("\n🚀 [Bắt đầu Training CNN 1D]")
    cnn_net = CNN1DModel(num_features=num_features, hidden_size=256, dropout=0.2)
    cnn_trainer = NNTrainer(cnn_net, lr=1e-3, weight_decay=1e-5)
    
    cnn_trainer.train(X_train, y_train, X_val, y_val, batch_size=2048, epochs=100, patience=12)
    
    cnn_val_preds = cnn_trainer.predict(X_val, batch_size=4096)
    cnn_oof_preds[val_idx] = cnn_val_preds
    cnn_trainer.save_model(os.path.join(MODELS_DIR, f'cnn/cnn_fold{fold+1}.pt'))
    
    
    # --- 2. Huấn luyện MLP ---
    print("\n🚀 [Bắt đầu Training MLP]")
    mlp_net = MLPModel(num_features=num_features, hidden_sizes=[512, 256, 128, 64], dropout=0.2)
    mlp_trainer = NNTrainer(mlp_net, lr=1e-3, weight_decay=1e-5)
    
    mlp_trainer.train(X_train, y_train, X_val, y_val, batch_size=2048, epochs=100, patience=12)
    
    mlp_val_preds = mlp_trainer.predict(X_val, batch_size=4096)
    mlp_oof_preds[val_idx] = mlp_val_preds
    mlp_trainer.save_model(os.path.join(MODELS_DIR, f'mlp/mlp_fold{fold+1}.pt'))
    
    # --- Điểm OOF Fold ---
    cnn_score = rmspe(y_val, cnn_val_preds)
    mlp_score = rmspe(y_val, mlp_val_preds)
    print(f"\n🏆 Fold {fold+1} RMSPE | CNN: {cnn_score:.5f} | MLP: {mlp_score:.5f}")
    
    del X_train, y_train, X_val, y_val, cnn_net, cnn_trainer, mlp_net, mlp_trainer
    torch.cuda.empty_cache()
    gc.collect()

### 3. Tổng Hợp Kết Quả & Đánh Giá Ensembling

In [ ]:
valid_indexes = cnn_oof_preds > 0
y_true_final = df.loc[valid_indexes, 'target'].values

final_cnn_rmspe = rmspe(y_true_final, cnn_oof_preds[valid_indexes])
final_mlp_rmspe = rmspe(y_true_final, mlp_oof_preds[valid_indexes])

# Ensemble CNN và MLP theo tỷ lệ 50/50 để xem sức mạnh của sự kết hợp
nn_ensemble_preds = (cnn_oof_preds[valid_indexes] + mlp_oof_preds[valid_indexes]) / 2.0
final_ensemble_rmspe = rmspe(y_true_final, nn_ensemble_preds)

print("="*50)
print(f"🔥 ĐIỂM SỐ CHUNG CUỘC CNN 1D      : {final_cnn_rmspe:.5f}")
print(f"🔥 ĐIỂM SỐ CHUNG CUỘC MLP         : {final_mlp_rmspe:.5f}")
print(f"🌟 ENSEMBLE CNN + MLP (50/50)    : {final_ensemble_rmspe:.5f}")
print("="*50)